In [ ]:
# Ensure Python Version 3.6 - 3.12!!!
!python --version

Python 3.11.13


In [ ]:
!pip install -q syncode

In [ ]:
from syncode import Syncode

# Define the grammar
We define the grammar for the problem we are solving. We consider the output to be constrained to two floats and an operator. We allow a new-line at the end to allow the model to "express" end of answer. This newline can be easily removed using .strip().

In [ ]:
grammar = r"""
    start: FLOAT OP FLOAT NL

    FLOAT: /(0|[1-9][0-9]{0,8})\.[0-9]{1,8}/
    OP: "+" | "-" | "*" | "/"
    NL: "\n"
"""

# Load the Huggingface models
Syncode uses the Huggingface transformers library to load the models. Simply provide the HuggingFace ID of the model to be used.

# Write the run_syncode function to do the operations.
We perform extraction using syncode and then use Python interpreter to solve the arithmetic. We provide few-shot examples.

In [ ]:
import re, operator

model_name = "microsoft/phi-2"
syn_llm = Syncode(model=model_name, mode='grammar_mask', grammar=grammar,
                  parse_output_only=True, max_new_tokens=50)

FEWSHOT = (
    "Extract the two numbers and the operator as floats.\n"
    "Q: What is 10.0 divided by 4.?\nA: 10.0/4.0\n"
    "Q: What is 8.25 minus 3.?\nA: 8.25-3.0\n"
    "Q: What is 6. multiplied by 1.5?\nA: 6.0*1.5\n"
    "Q: What is 48.9 plus 19.86?\nA: 48.9+19.86\n"
)

OPS = {"+": operator.add, "-": operator.sub,
       "*": operator.mul, "/": operator.truediv}

def run_syncode(textual_query):
    prompt = f"{FEWSHOT}Q: {textual_query}\nA: "
    out = syn_llm.infer(prompt)[0].strip()          # e.g. "327.0*11.0"

    m = re.fullmatch(r"(\d+\.\d+)([+\-*/])(\d+\.\d+)", out)
    if m is None:
        raise ValueError(f"Unexpected SynCode output: {out!r}")

    a, op, b = float(m.group(1)), m.group(2), float(m.group(3))
    answer = float(OPS[op](a, b))                   # Python does the math
    return ([a, b], op, answer)

print(run_syncode("What is 327. multiplied by 11.0?"))
print(run_syncode("What is 45.1 plus 23.54?"))
print(run_syncode("What is 120.4 divided by 4.0?"))
print(run_syncode("What is 90.5 minus 12.25?"))
print(run_syncode("Multiply 3.2 by 4.0."))


[2026-10-27 04:50:53,484-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16
[2026-10-27 04:50:53,785-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


([327.0, 11.0], '*', 3597.0)
([45.1, 23.54], '+', 68.64)
([120.4, 4.0], '/', 30.1)
([90.5, 12.25], '-', 78.25)
([3.2, 4.0], '*', 12.8)
